# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Additional Notes**
- Using sim_year as the actual year 

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

---
**!!! ToDo**
- plot model fits 
- make flow diagram of model
- (?) create a mapping function for location info and lat/lon... store as JSON to save lookup
- add Confidence Intervals for Return Levels
- add return level (and CI) to txt
- maximum likelihood method or Bayesian? ... delta method?!<br>
✓ time-series for stationary analysis (ignore shape and scale from global analysis)<br>
✓ check why slope $\mu(t)$ != $trend$ (m/year) -> mistake trend was calculated as $\mu(t)*years$<br>
✓ declutter what to save!<br>
✓ store report output as parquet / Feather


# Import Libraries

In [ ]:
import sys
import random
import time
from datetime import datetime
from glob import glob

import xarray as xr
from IPython.display import Markdown, display
from pandas import DataFrame

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

# Settings

In [3]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [4]:
hindcast_start = 1960
hindcast_end = 2026

In [5]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [6]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [7]:
print_msg = False
display_results = False
export_report=True
save_regression_summary = True

# Import data

In [8]:
ls_files = [file for file in glob(path + '*.nc')]
ls_files

['../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc']

In [9]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

Importing data from model MIROC6 (1/8)...
Importing data from model MPI-ESM1-2-HR (2/8)...
Importing data from model HadGEM3-GC31-MM (3/8)...
Importing data from model MRI-ESM2-0 (4/8)...
Importing data from model BCC-CSM2-MR (5/8)...
Importing data from model CMCC-CM2-SR5 (6/8)...
Importing data from model CanESM5 (7/8)...
Importing data from model NorCPM1 (8/8)...


# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, using `joblib` - saving ~60% (from 3min30sec down to 1min22sec)


In [10]:
time_start1 = datetime.now()
dic_data_per_model = dbf.data_preparation(ls_files=ls_files, dic_data_per_model=dic_data_per_model)
time_end1 = datetime.now()

In [11]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))
ut.create_data_overview(dic_data_per_model,ls_files)

display(Markdown(f"Execution time · {time_end1 - time_start1}sec"))

**Data Overview**

**Model · model shape: samples (~sim_years) | ensemble members | valid locations**

MIROC6 · (680, 2, 7054)
MPI-ESM1-2-HR · (630, 2, 5808)
HadGEM3-GC31-MM · (680, 2, 7216)
MRI-ESM2-0 · (320, 2, 3547)
BCC-CSM2-MR · (630, 2, 5236)
CMCC-CM2-SR5 · (690, 2, 6038)
CanESM5 · (660, 2, 5782)
NorCPM1 · (630, 2, 4558)

Overall, data is available from 
	8 models, 
	3547-7216 locations (originally 11022)
	320-690 samples per model 
	 - with 63-69 unique sim_years
	 - between 1961-2029


Execution time · 0:01:26.883598sec

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)


In [12]:
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
print("Final dimensions:", combined.dims)
print("Shape:", combined.shape)
print("Number of models:", combined.model.size)
print("Number of locations:", combined.location.size)

/var/folders/lx/z70mzvpx4ls9np3hfbhll4wr0000gn/T/ipykernel_14516/325246176.py:10: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  combined = xr.concat(da_list, dim="model", join="outer")



**Overall, the combined dataset has the following dimensions**

Final dimensions: ('model', 'sample', 'member', 'location')
Shape: (8, 690, 2, 9589)
Number of models: 8
Number of locations: 9589


### Validation Check

In [13]:
list_model_labels = list(dic_data_per_model.keys())

In [14]:
model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

For validity check, use randomly selected model CanESM5 and site-id 3363...



**Overview Original dataArray**

Model CanESM5 (None) - location-ID 3363 
Full dataframe (660, 2) vs reduced (567, 2)
coordinates in original dataset lon|lat: 27.21945|36.82082



**Overview Revised dataArray**

Model None (6) - location-ID None 
Full dataframe (690, 2) vs reduced (567, 2)
coordinates in original dataset lon|lat: 27.21945|36.82082


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:258: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [15]:
assert lon_target == lon_rev and lat_target == lat_rev
assert all(data_for_model_for_location.dropna() == revised_dataset.dropna())

**Learning** · make sure to never select by the site-id but always via geo-coordinates (lat | lon)

# Workflow GEV - Generalized Extreme Value

## OPTION1
Using all data available per location - from all years and models

#### Initial trial with subset of TWO locations
Later, batch(?) and parallelize

In [28]:
list_sites = [42, 111, 1919, 7012, 9000]

In [29]:
dic_data_per_location = {}
for loc_ex in list_sites: 
    data_at_location = combined[:,:,:, loc_ex].to_dataframe().dropna().reset_index()
    data_at_location = data_at_location.rename(columns={'annualMax':'storm_surge'})
    dic_data_per_location[loc_ex] = data_at_location

### Run Analysis

Note, the output is stored as 
- visuals → png
- tabular data (DataFrames) → Parquet
- other objects (dicts, strings, floats) → Pickle

File structure
```results/
├─ location_1/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
├─ location_2/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
...
```

In [30]:
orig_stdout, orig_stderr, fh, logger, log_path = ut.initialize_logger(
    f"LOGS_GEVAnalysis_pooled_{datetime.now():%Y%m%d_%H%M%S}.log"
    )

# ------------------------------------------------------------------------------------------
time_start = time.time()

print("\n" + "="*100)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER LOCATION")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_prepared = ut.prepare_pooled_data(
    dic_data=dic_data_per_location,
    hindcast_start=hindcast_start,
    hindcast_end=hindcast_end
)

# ------------------------------------------------------------------------------------------
results = {}
for loc_id, df_prepared in dic_prepared.items():
    print("\n" + "-"*70)
    print(f"Analyzing location id {loc_id} ...")

    lon_loc = df_prepared.lon.unique()[0]
    lat_loc = df_prepared.lat.unique()[0]

    print("\tLookup location info...")
    location_info = dbf.locations_label_lookup_simple(lon=lon_loc, lat=lat_loc)
    print(f"\t → Closest location identified: {location_info}")

    result = gev.analyze_per_location(
        df_prepared, loc_id, lat_loc, lon_loc, location_info, return_periods
    )
    if result is None:
        print(f"\t→ Warning! No valid GEV fit for location id {loc_id}. Skipping ...")
        continue
    
    if export_report and path_export: 
        export_path_site = ut.save_location_results(            
            location_id=loc_id, result_location=result, base_dir=path_export, 
            plot_period_evolution=plot_period_evolution, display_results=display_results
            )
        
    result['file_path_report'] = export_path_site
    results[loc_id] = result

# ------------------------------------------------------------------------------------------
time_end = time.time()
print("\n" + "="*100)
print(f"✓ ANALYSIS COMPLETED IN {(time_end - time_start):.2f}s!")
print("="*100)

# ------------------------------------------------------------------------------------------
sys.stdout = orig_stdout
sys.stderr = orig_stderr
logger.removeHandler(fh)
fh.close()

Upscaling to 11022 locations, will result in an execution time of ~8.7hours!

## OPTION2
Re-run stationary GEV per year (for location parameter; scale and shape remain as globally defined)

In [31]:
# del results
try:
    results.keys()
    print('continue with available dictionary')
    
except NameError:
    print('import data from files')
    results = ut.import_results_from_files(path_export)

continue with available dictionary


In [32]:
time_start3 = datetime.now()
results_extended = gev.execute_and_store_stat_gev_per_year(results=results, store_results=False)
time_start4 = datetime.now()

# --------------------------------------------------------------------------
time_diff = time_start4 - time_start3
print(f"Execution time for computing GEV per year: {time_diff}sec")

Conducting stationary GEV for siteID 42 grouped per year...
	...2026
Done!

Conducting stationary GEV for siteID 111 grouped per year...
	...2026
Done!

Conducting stationary GEV for siteID 1919 grouped per year...
	...2026
Done!

Conducting stationary GEV for siteID 7012 grouped per year...
	...2026
Done!

Conducting stationary GEV for siteID 9000 grouped per year...
	...2026
Done!

Execution time for computing GEV per year: 0:00:18.403878sec


Upscaling to 11022 locations, will result in an execution time of ~12.5hours!

---
to be continued

### Regression of location parameter over years
include uncertainty given by n_obs

**NOTE**<br>
X = sm.add_constant(df['year']) 
>> centering the year parameter due to the following warning:
"The condition number is large, 2.37e+05. This might indicate that there are strong multicollinearity or other numerical problems."

In [ ]:
for site_id, dic_location in results_extended.items():
    df_stat_gev_per_year = dic_location['fit results']['gev_stationary']['analysis_per_year']
    df = df_stat_gev_per_year.reset_index().rename(columns={'index': 'year'})


,shape,location,scale,n_obs,log_likelihood,aic,bic,dist_type,tail_behavior
1961,-0.152823,0.264709,0.046098,12,18.867038,-31.734076,-30.279356,Weibull (Type III),Light (bounded)
1962,0.053326,0.248811,0.039478,24,38.971785,-71.943571,-68.409409,Fréchet (Type II),Heavy (polynomial)
1963,-0.251943,0.268701,0.055719,36,52.513757,-99.027514,-94.276958,Weibull (Type III),Light (bounded)
1964,-1.396801,0.278198,0.134551,46,58.256474,-110.512947,-105.027023,Weibull (Type III),Light (bounded)
1965,-0.06969,0.262773,0.047724,57,85.848289,-165.696577,-159.567424,Weibull (Type III),Light (bounded)
...,...,...,...,...,...,...,...,...,...
2022,-0.18972,0.246842,0.049124,76,118.739915,-231.479829,-224.487629,Weibull (Type III),Light (bounded)
2023,-0.147995,0.255954,0.046368,64,101.468395,-196.936789,-190.46014,Weibull (Type III),Light (bounded)
2024,-0.018694,0.248757,0.041015,50,81.89932,-157.79864,-152.062571,Gumbel (Type I),Exponential
2025,-0.078791,0.262864,0.046861,38,58.72133,-111.44266,-106.529902,Weibull (Type III),Light (bounded)


In [ ]:
for site_id, dic_location in results_extended.items():
    df_stat_gev_per_year = dic_location['fit results']['gev_stationary']['analysis_per_year']
    df = df_stat_gev_per_year.reset_index().rename(columns={'index': 'year'})
    
    global_statgev_shape = dic_location['fit results']['gev_stationary']['shape']
    global_statgev_scale = dic_location['fit results']['gev_stationary']['scale']

    [
        wls_delta, weights, y_pred, year_grid
        ] = gev.weighted_least_square_regression_annual_location(global_statgev_scale, global_statgev_shape, df)
    
    fig = dbplt.plot_gev_mu_trend(
        df=df_stat_gev_per_year,
        weights=weights,
        year_grid=year_grid,
        y_pred=y_pred,
        wls_delta=wls_delta,
        display_results=display_results,
        )
    
    if save_regression_summary:
        save_path = dic_location['file_path_report']
        with open(save_path + '/WLSdelta_summary.html', 'w') as f:
            f.write( wls_delta.summary().as_html())
        
        lat = location_info['lat']
        lon = location_info['lon']
        country = location_info['description'][0].split(',')[-1].strip()  
        file_name = f"/GEVTrendAnalysis_location_{str(site_id)}_{country}_{lat}|{lon}.png"
        fig.savefig(save_path+file_name, dpi=300, bbox_inches='tight')
        print(f"\t saving GEV μ trend analysis to {save_path}.")


TypeError: plot_gev_mu_trend() got an unexpected keyword argument 'site_id'